# scTranslator Predictions — zero-shot inference for the eNet comparison

This notebook **computes the scTranslator protein-abundance predictions** that
`02. enet_scT_comparison.ipynb` loads for the ElasticNet-vs-scTranslator comparison.
It is a direct copy of the benchmarking pipeline that was used to generate the
`res_data/sc_predictions_{2M,160K,10K}.npz` bundles.

For each publicly available scTranslator checkpoint (**2M**, **160K**, **10K**) it:
1. loads the matched transcriptomics / proteomics `.joblib` tables,
2. handles NaNs (drop columns >50% NaN, fill the rest with `1e-8`),
3. maps gene / protein symbols to scTranslator vocabulary IDs,
4. applies the paper's per-cell min-max normalization,
5. runs zero-shot inference (proteins are chunked to the decoder's 1000 limit), and
6. saves one `sc_predictions_<checkpoint>.npz` bundle per checkpoint into `res_data/`.

Each bundle holds `predictions` (cells × proteins), `protein_names`, `protein_ids`
and `cell_names` — exactly the keys `02. enet_scT_comparison.ipynb` expects.

**Prerequisites (not in this repo — see README):**
- the scTranslator source repo (for `performer_enc_dec.scPerformerEncDec`),
- the three checkpoint files `scTranslator_{2M,160K,10K}.pt`,
- the ID dictionaries under `code/model/ID_dic`,
- the matched input tables `transcriptomics_filtered.joblib` and `proteomics_matched.joblib`.

## 1. Imports

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "03. compute_sctranslator_predictions"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
import os
import sys
import pickle
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 2. Paths & Configuration

All paths can be overridden with environment variables so the notebook runs
unchanged on a cluster. The defaults below are the local paths that were used to
produce the committed prediction bundles — **edit them for your machine**.

In [ ]:
# --- scTranslator code + checkpoints (NOT in this repo) ---------------------
# The scTranslator repo provides code/model/performer_enc_dec.py and code/model/ID_dic/.
def _find_repo_root(start=Path.cwd().resolve()):
    for p in [start, *start.parents]:
        if (p / '06. validation').is_dir() and (p / '01. pre-processing').is_dir():
            return p
    return start
REPO_ROOT = _find_repo_root()
SCTRANSLATOR_REPO = os.environ.get(
    'SCTRANSLATOR_REPO', str(REPO_ROOT / 'external' / 'scTranslator' / 'scTranslator'))
# Folder holding scTranslator_2M.pt / scTranslator_160K.pt / scTranslator_10K.pt
CHECKPOINT_DIR = os.environ.get(
    'SCTRANSLATOR_CKPT_DIR', str(REPO_ROOT / 'external' / 'scTranslator' / 'checkpoints'))

# --- Input data (matched transcriptomics / proteomics tables) --------------
# rows = samples, columns = gene / protein symbols. Same tables the eNet pipeline uses.
DATA_DIR  = Path(os.environ.get('CPRR_DATA_DIR', 'data'))
RNA_PATH  = Path(os.environ.get('CPRR_RNA_PATH',  DATA_DIR / 'transcriptomics_filtered.joblib'))
PROT_PATH = Path(os.environ.get('CPRR_PROT_PATH', DATA_DIR / 'proteomics_matched.joblib'))

# --- Output: 02. enet_scT_comparison.ipynb reads res_data/sc_predictions_*.npz ---
OUTPUT_DIR = Path(os.environ.get('CPRR_PRED_OUT', 'res_data'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Make the scTranslator model importable
sys.path.append(os.path.join(SCTRANSLATOR_REPO, 'code', 'model'))
from performer_enc_dec import scPerformerEncDec  # noqa: F401  (needed to unpickle the checkpoints)

CONFIG = {
    'rna_path':  str(RNA_PATH),
    'prot_path': str(PROT_PATH),
    'dic_path':  os.path.join(SCTRANSLATOR_REPO, 'code', 'model', 'ID_dic'),
    'checkpoints': {
        '2M':   os.path.join(CHECKPOINT_DIR, 'scTranslator_2M.pt'),
        '160K': os.path.join(CHECKPOINT_DIR, 'scTranslator_160K.pt'),
        '10K':  os.path.join(CHECKPOINT_DIR, 'scTranslator_10K.pt'),
    },
    'enc_max_seq_len': 20000,
    'dec_max_seq_len': 1000,   # decoder limit -> proteins are chunked to this size
    'batch_size': 8,
    'output_dir': str(OUTPUT_DIR),
}

print(f"RNA_PATH:   {RNA_PATH}  [{'FOUND' if RNA_PATH.exists() else 'MISSING'}]")
print(f"PROT_PATH:  {PROT_PATH}  [{'FOUND' if PROT_PATH.exists() else 'MISSING'}]")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
for name, path in CONFIG['checkpoints'].items():
    print(f"  [{'FOUND' if os.path.exists(path) else 'MISSING'}] {name}: {path}")

## 3. ID Mapping Helpers

In [ ]:
def load_id_dictionaries(dic_path):
    """Load scTranslator ID mapping dictionaries."""
    with open(os.path.join(dic_path, 'hgs_to_EntrezID.pkl'), 'rb') as f:
        hgs_to_entrez = pickle.load(f)
    with open(os.path.join(dic_path, 'EntrezID_to_myID.pkl'), 'rb') as f:
        entrez_to_myid = pickle.load(f)
    return hgs_to_entrez, entrez_to_myid


def gene_symbol_to_myid(gene_symbol, hgs_to_entrez, entrez_to_myid):
    """Convert a gene symbol to a scTranslator vocabulary ID."""
    entrez_id = hgs_to_entrez.get(gene_symbol)
    if entrez_id:
        my_id = entrez_to_myid.get(entrez_id)
        return int(my_id) if my_id else None
    return None


def map_genes_to_ids(gene_list, hgs_to_entrez, entrez_to_myid):
    """Map a list of gene symbols to scTranslator IDs."""
    mapped, unmapped = [], []
    for gene in gene_list:
        my_id = gene_symbol_to_myid(gene, hgs_to_entrez, entrez_to_myid)
        if my_id is not None:
            mapped.append((gene, my_id))
        else:
            unmapped.append(gene)
    return mapped, unmapped

## 4. Normalization

In [ ]:
def cellwise_minmax(X, low=1e-8, high=1.0):
    """Paper's normalization: per-cell min-max scaling to [low, high]."""
    X = X.astype(np.float32)
    mins = X.min(axis=1, keepdims=True)
    maxs = X.max(axis=1, keepdims=True)
    denom = (maxs - mins)
    denom[denom == 0] = 1.0  # avoid division by zero
    Xn = (X - mins) / denom
    Xn = Xn * (high - low) + low
    return Xn.astype(np.float32)

## 5. Data Preparation (NaN handling + ID mapping)

- Columns with **>50% NaN** across samples are **dropped**.
- Remaining NaN values are **filled with `1e-8`** (0 is scTranslator's special prediction token).
- Symbols that don't map to the scTranslator vocabulary are discarded.

In [ ]:
def prepare_data(config):
    """Load and prepare RNA + protein tables with NaN handling and ID mapping."""
    NAN_THRESHOLD = 0.50  # drop a column if more than 50% of samples are NaN

    print('Loading data...')
    rna_df  = joblib.load(config['rna_path'])
    prot_df = joblib.load(config['prot_path'])

    rna_orig_cols  = len(rna_df.columns)
    prot_orig_cols = len(prot_df.columns)
    n_samples_rna  = len(rna_df)
    n_samples_prot = len(prot_df)

    print(f'RNA shape (raw):     {rna_df.shape}')
    print(f'Protein shape (raw): {prot_df.shape}')

    # --- NaN handling: drop if >50% NaN, else fill with 1e-8 ---
    rna_nan_frac  = rna_df.isna().sum() / n_samples_rna
    prot_nan_frac = prot_df.isna().sum() / n_samples_prot

    rna_drop_cols  = rna_nan_frac[rna_nan_frac > NAN_THRESHOLD].index.tolist()
    prot_drop_cols = prot_nan_frac[prot_nan_frac > NAN_THRESHOLD].index.tolist()

    rna_df  = rna_df.drop(columns=rna_drop_cols).fillna(1e-8)
    prot_df = prot_df.drop(columns=prot_drop_cols).fillna(1e-8)

    print(f'\nAfter NaN handling (>{NAN_THRESHOLD*100:.0f}% -> drop, else fill 1e-8):')
    print(f'  RNA:     {rna_orig_cols} -> {rna_df.shape[1]} genes')
    print(f'  Protein: {prot_orig_cols} -> {prot_df.shape[1]} proteins')

    # --- ID mapping ---
    hgs_to_entrez, entrez_to_myid = load_id_dictionaries(config['dic_path'])

    rna_mapped,  rna_unmapped  = map_genes_to_ids(rna_df.columns.tolist(),  hgs_to_entrez, entrez_to_myid)
    prot_mapped, prot_unmapped = map_genes_to_ids(prot_df.columns.tolist(), hgs_to_entrez, entrez_to_myid)

    print('\nscTranslator vocabulary mapping:')
    print(f"  RNA genes mapped: {len(rna_mapped)}/{len(rna_df.columns)} "
          f"({100*len(rna_mapped)/max(len(rna_df.columns),1):.1f}%), unmapped {len(rna_unmapped)}")
    print(f"  Proteins mapped:  {len(prot_mapped)}/{len(prot_df.columns)} "
          f"({100*len(prot_mapped)/max(len(prot_df.columns),1):.1f}%), unmapped {len(prot_unmapped)}")
    if prot_unmapped:
        print(f'  Unmapped proteins: {prot_unmapped}')

    rna_genes  = [g for g, _ in rna_mapped]
    rna_ids    = [i for _, i in rna_mapped]
    prot_genes = [g for g, _ in prot_mapped]
    prot_ids   = [i for _, i in prot_mapped]

    rna_values  = rna_df[rna_genes].values
    prot_values = prot_df[prot_genes].values

    print('\nApplying cell-wise min-max normalization...')
    rna_norm  = cellwise_minmax(rna_values)
    prot_norm = cellwise_minmax(prot_values)

    return {
        'rna_values':  rna_norm,
        'rna_ids':     np.array(rna_ids),
        'rna_genes':   rna_genes,
        'prot_values': prot_norm,
        'prot_ids':    np.array(prot_ids),
        'prot_genes':  prot_genes,
        'cell_names':  rna_df.index.tolist(),
        'n_cells':     rna_df.shape[0],
        'n_rna_genes': len(rna_genes),
        'n_proteins':  len(prot_genes),
    }

## 6. Model Inference

In [ ]:
def prepare_batch(rna_values, rna_ids, prot_ids, enc_max_len, dec_max_len, device):
    """Pad / truncate encoder (RNA) and decoder (protein) sequences for a batch."""
    batch_size = rna_values.shape[0]
    n_rna  = rna_values.shape[1]
    n_prot = len(prot_ids)

    # Encoder (RNA) - truncate or pad to enc_max_len
    if n_rna >= enc_max_len:
        rna_val  = rna_values[:, :enc_max_len]
        rna_id   = np.tile(rna_ids[:enc_max_len], (batch_size, 1))
        rna_mask = np.ones((batch_size, enc_max_len), dtype=bool)
    else:
        rna_val = np.zeros((batch_size, enc_max_len), dtype=np.float32)
        rna_val[:, :n_rna] = rna_values
        rna_id = np.zeros((batch_size, enc_max_len), dtype=np.int64)
        rna_id[:, :n_rna] = rna_ids
        rna_mask = np.zeros((batch_size, enc_max_len), dtype=bool)
        rna_mask[:, :n_rna] = True

    # Decoder (Protein) - pad to dec_max_len (<= 1000)
    prot_val  = np.zeros((batch_size, dec_max_len), dtype=np.float32)
    prot_id   = np.zeros((batch_size, dec_max_len), dtype=np.int64)
    prot_mask = np.zeros((batch_size, dec_max_len), dtype=bool)
    prot_id[:, :n_prot]   = prot_ids
    prot_mask[:, :n_prot] = True

    rna_val_t   = torch.tensor(rna_val,  dtype=torch.float32).to(device)
    rna_id_t    = torch.tensor(rna_id,   dtype=torch.long).to(device)
    rna_mask_t  = torch.tensor(rna_mask, dtype=torch.bool).to(device)
    prot_id_t   = torch.tensor(prot_id,  dtype=torch.long).to(device)
    prot_mask_t = torch.tensor(prot_mask, dtype=torch.bool).to(device)
    return rna_val_t, rna_id_t, rna_mask_t, prot_id_t, prot_mask_t, n_prot


def run_inference(model, data, config, device):
    """Run inference over all cells, chunking proteins to the decoder limit."""
    model.eval()
    n_cells    = data['n_cells']
    n_proteins = data['n_proteins']
    dec_max    = config['dec_max_seq_len']
    batch_size = config['batch_size']

    n_chunks = int(np.ceil(n_proteins / dec_max))
    print(f'\nProtein chunking: {n_proteins} proteins -> {n_chunks} chunk(s) of <={dec_max}')

    all_predictions = np.zeros((n_cells, n_proteins), dtype=np.float32)

    for chunk_idx in range(n_chunks):
        start_prot = chunk_idx * dec_max
        end_prot   = min((chunk_idx + 1) * dec_max, n_proteins)
        chunk_prot_ids = data['prot_ids'][start_prot:end_prot]
        chunk_size = end_prot - start_prot
        print(f'  Chunk {chunk_idx+1}/{n_chunks}: proteins {start_prot}-{end_prot} ({chunk_size})')

        chunk_predictions = []
        for batch_start in tqdm(range(0, n_cells, batch_size), desc=f'  Chunk {chunk_idx+1}'):
            batch_end = min(batch_start + batch_size, n_cells)
            batch_rna = data['rna_values'][batch_start:batch_end]

            rna_val, rna_id, rna_mask, prot_id, prot_mask, n_prot = prepare_batch(
                batch_rna, data['rna_ids'], chunk_prot_ids,
                config['enc_max_seq_len'], dec_max, device,
            )
            with torch.no_grad():
                _, y_hat = model(rna_val, rna_id, prot_id, enc_mask=rna_mask, dec_mask=prot_mask)
                y_hat = torch.squeeze(y_hat)
                if len(y_hat.shape) == 1:
                    y_hat = y_hat.unsqueeze(0)
                y_hat_valid = y_hat[:, :chunk_size].cpu().numpy()
                chunk_predictions.append(y_hat_valid)

        all_predictions[:, start_prot:end_prot] = np.vstack(chunk_predictions)

    return all_predictions

## 7. Metrics (optional sanity check)

The full comparison / plotting lives in `02. enet_scT_comparison.ipynb`; these are just
quick per-cell / per-protein numbers so you can confirm a run looks sane.

In [ ]:
def compute_metrics(y_true, y_pred):
    """Per-cell cosine, PCC, MSE, MAE (returned as arrays)."""
    cosines, pccs, mses, maes = [], [], [], []
    for i in range(y_true.shape[0]):
        t, p = y_true[i], y_pred[i]
        nt, npd = np.linalg.norm(t), np.linalg.norm(p)
        cosines.append(np.dot(t, p) / (nt * npd) if (nt > 0 and npd > 0) else 0.0)
        pccs.append(pearsonr(t, p)[0] if (np.std(t) > 0 and np.std(p) > 0) else 0.0)
        mses.append(mean_squared_error(t, p))
        maes.append(mean_absolute_error(t, p))
    return {'cosine': np.array(cosines), 'pcc': np.array(pccs),
            'mse': np.array(mses), 'mae': np.array(maes)}


def compute_protein_wise_pcc(y_true, y_pred):
    """PCC per protein (across all cells)."""
    pccs = []
    for j in range(y_true.shape[1]):
        t, p = y_true[:, j], y_pred[:, j]
        pccs.append(pearsonr(t, p)[0] if (np.std(t) > 0 and np.std(p) > 0) else 0.0)
    return np.array(pccs)

## 8. Compute predictions for one checkpoint

Saves `res_data/sc_predictions_<checkpoint>.npz` with keys
`predictions`, `protein_names`, `protein_ids`, `cell_names` — the exact bundle
`02. enet_scT_comparison.ipynb` loads.

In [ ]:
def predict_checkpoint(checkpoint_name, checkpoint_path, data, config, device):
    print(f"\n{'='*60}\nCheckpoint: {checkpoint_name}\n{checkpoint_path}\n{'='*60}")

    print('Loading model...')
    # The checkpoints are full pickled scPerformerEncDec objects, so weights_only=False
    # is required on PyTorch >= 2.6 (default flipped to True). Safe here: it's our own checkpoint.
    try:
        model = torch.load(checkpoint_path, map_location=device, weights_only=False)
    except TypeError:  # PyTorch < 2.6 has no weights_only kwarg
        model = torch.load(checkpoint_path, map_location=device)
    model.eval()

    predictions = run_inference(model, data, config, device)

    expected = (data['n_cells'], data['n_proteins'])
    if predictions.shape != expected:
        raise ValueError(f'Shape mismatch: got {predictions.shape}, expected {expected}')

    out_path = os.path.join(config['output_dir'], f'sc_predictions_{checkpoint_name}.npz')
    np.savez(
        out_path,
        predictions=predictions,
        protein_names=np.array(data['prot_genes']),
        protein_ids=np.array(data['prot_ids']),
        cell_names=np.array(data['cell_names']),
    )
    print(f'  Saved: {out_path}')

    metrics = compute_metrics(data['prot_values'], predictions)
    prot_pcc = compute_protein_wise_pcc(data['prot_values'], predictions)
    print(f"  Per-cell   PCC: {metrics['pcc'].mean():.4f} | Cosine: {metrics['cosine'].mean():.4f} "
          f"| MSE: {metrics['mse'].mean():.6f}")
    print(f"  Per-protein PCC: {prot_pcc.mean():.4f}")
    return {'predictions': predictions, 'metrics': metrics, 'protein_pccs': prot_pcc}

---
## 9. Run

The cells above only defined helpers. Everything below actually executes.

### 9.1 Device

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

### 9.2 Prepare data (once, shared across checkpoints)

In [ ]:
data = prepare_data(CONFIG)

### 9.3 Run all checkpoints

Runtime scales with cells × proteins; on CPU this is slow. Comment out checkpoints
in `CONFIG['checkpoints']` to run a single one (e.g. keep only `10K` for a smoke test).

In [ ]:
all_results = {}
for ckpt_name, ckpt_file in CONFIG['checkpoints'].items():
    if os.path.exists(ckpt_file):
        all_results[ckpt_name] = predict_checkpoint(ckpt_name, ckpt_file, data, CONFIG, device)
        if device == 'cuda':
            torch.cuda.empty_cache()
    else:
        print(f'\nWARNING: checkpoint not found, skipping: {ckpt_file}')

print('\nDone. Wrote:', sorted(p.name for p in Path(CONFIG['output_dir']).glob('sc_predictions_*.npz')))